In [ ]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Board configuration
FINISH = 34

# Snakes and ladders transitions
transitions = {
    # Ladders (move up)
    1: 12,
    5: 16,
    11: 22,
    15: 23,
    20: 31,
    # Snakes (move down)
    7: 4,
    10: 2,
    21: 13,
    24: 6,
    33: 19,
}

# Identify which transitions are snakes (go down)
snakes = {k: v for k, v in transitions.items() if v < k}

def apply_transition(pos):
    """Apply snake or ladder transition if player lands on one."""
    return transitions.get(pos, pos)

print("Board setup complete.")
print(f"Finish square: {FINISH}")
print(f"Ladders: {[(k,v) for k,v in transitions.items() if v > k]}")
print(f"Snakes: {[(k,v) for k,v in transitions.items() if v < k]}")

In [ ]:
def simulate_single_player(num_games=5000):
    """Simulate single-player games.
    Returns list of number of rolls needed to finish each game.
    """
    rolls_list = []
    for _ in range(num_games):
        pos = 0
        rolls = 0
        while pos < FINISH:
            die = np.random.randint(1, 7)
            rolls += 1
            pos += die
            if pos >= FINISH:
                break
            pos = apply_transition(pos)
        rolls_list.append(rolls)
    return rolls_list


def simulate_two_player(num_games=5000, p2_start=0, p2_snake_immunity=False):
    """Simulate two-player games.
    
    Args:
        num_games: Number of games to simulate.
        p2_start: Starting square for Player 2.
        p2_snake_immunity: If True, Player 2 is immune to the first snake.
    
    Returns:
        Tuple of (p1_wins count, list of total combined rolls per game).
    """
    p1_wins = 0
    total_rolls_list = []
    for _ in range(num_games):
        pos = [0, p2_start]
        total_rolls = 0
        p2_used_immunity = False
        game_over = False
        while not game_over:
            for player in range(2):
                die = np.random.randint(1, 7)
                total_rolls += 1
                pos[player] += die
                if pos[player] >= FINISH:
                    if player == 0:
                        p1_wins += 1
                    game_over = True
                    break
                new_pos = apply_transition(pos[player])
                # Check if Player 2 has snake immunity
                if (player == 1 and p2_snake_immunity
                        and not p2_used_immunity
                        and new_pos < pos[player]):
                    # This is a snake - use immunity, don't move
                    p2_used_immunity = True
                else:
                    pos[player] = new_pos
        total_rolls_list.append(total_rolls)
    return p1_wins, total_rolls_list

print("Simulation functions defined.")

In [ ]:
# Question 1: Average rolls for a single player to finish
# Run multiple batches of 5000 games for stability
NUM_BATCHES = 20
GAMES_PER_BATCH = 5000

q1_averages = []
for _ in range(NUM_BATCHES):
    rolls = simulate_single_player(GAMES_PER_BATCH)
    q1_averages.append(np.mean(rolls))

q1_avg = np.mean(q1_averages)
print(f"Q1: Average rolls for single player = {q1_avg:.2f}")

# Options: a=7, b=9, c=11, d=13
q1_options = {7: 'A', 9: 'B', 11: 'C', 13: 'D'}
q1_closest = min(q1_options.keys(), key=lambda x: abs(x - q1_avg))
q1_answer = q1_options[q1_closest]
print(f"Q1 closest option: {q1_answer} ({q1_closest} rolls)")

In [ ]:
# Question 2: Average combined rolls in a two-player game
# Question 3: Probability that Player 1 wins

q2_averages = []
q3_win_rates = []
for _ in range(NUM_BATCHES):
    p1_wins, total_rolls = simulate_two_player(GAMES_PER_BATCH)
    q2_averages.append(np.mean(total_rolls))
    q3_win_rates.append(p1_wins / GAMES_PER_BATCH)

q2_avg = np.mean(q2_averages)
print(f"Q2: Average combined rolls = {q2_avg:.2f}")
q2_options = {13: 'A', 15: 'B', 17: 'C', 19: 'D'}
q2_closest = min(q2_options.keys(), key=lambda x: abs(x - q2_avg))
q2_answer = q2_options[q2_closest]
print(f"Q2 closest option: {q2_answer} ({q2_closest} rolls)")

q3_prob = np.mean(q3_win_rates) * 100
print(f"\nQ3: P(Player 1 wins) = {q3_prob:.2f}%")
q3_options = {50: 'A', 53: 'B', 57: 'C', 60: 'D'}
q3_closest = min(q3_options.keys(), key=lambda x: abs(x - q3_prob))
q3_answer = q3_options[q3_closest]
print(f"Q3 closest option: {q3_answer} ({q3_closest}%)")

In [ ]:
# Question 4: Which starting square for P2 gives closest to 50/50 odds?
# Options: a=3, b=6, c=9, d=12

q4_options_map = {3: 'A', 6: 'B', 9: 'C', 12: 'D'}
best_sq = None
best_diff = float('inf')

for start_sq in [3, 6, 9, 12]:
    win_rates = []
    for _ in range(NUM_BATCHES):
        p1_wins, _ = simulate_two_player(GAMES_PER_BATCH, p2_start=start_sq)
        win_rates.append(p1_wins / GAMES_PER_BATCH)
    avg_rate = np.mean(win_rates) * 100
    diff = abs(avg_rate - 50)
    print(f"  P2 starts at square {start_sq}: P(P1 wins) = {avg_rate:.2f}% (diff from 50% = {diff:.2f}%)")
    if diff < best_diff:
        best_diff = diff
        best_sq = start_sq

q4_answer = q4_options_map[best_sq]
print(f"\nQ4 answer: {q4_answer} (Square {best_sq}, closest to 50/50)")

In [ ]:
# Question 5: P2 immune to first snake - what is P1's win probability?

q5_win_rates = []
for _ in range(NUM_BATCHES):
    p1_wins, _ = simulate_two_player(GAMES_PER_BATCH, p2_snake_immunity=True)
    q5_win_rates.append(p1_wins / GAMES_PER_BATCH)

q5_prob = np.mean(q5_win_rates) * 100
print(f"Q5: P(Player 1 wins) with P2 snake immunity = {q5_prob:.2f}%")
q5_options = {42.5: 'A', 46.5: 'B', 49.5: 'C', 52.5: 'D'}
q5_closest = min(q5_options.keys(), key=lambda x: abs(x - q5_prob))
q5_answer = q5_options[q5_closest]
print(f"Q5 closest option: {q5_answer} ({q5_closest}%)")

In [ ]:
# Final answers
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
}

print("Final answers:")
for k, v in answers.items():
    print(f"  {k}: {v}")